In [ ]:
import json
from urllib.parse import urlparse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# SETUP
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

# FILTER ROLE (UMUM INDUSTRI)
COMMON_ROLES = [
    "Frontend", "Backend", "Full Stack", "DevOps",
    "Data Analyst", "Data Scientist", "Data Engineer",
    "Machine Learning", "AI Engineer", "Cyber Security",
    "QA", "UX Design"
]

def is_common_role(title):
    return any(role.lower() in title.lower() for role in COMMON_ROLES)

# FILTER URL (ROADMAP UTAMA SAJA)
def is_valid_roadmap_url(url):
    try:
        path = urlparse(url).path.strip("/")

        # hanya 1 level path (contoh: /frontend)
        if "/" in path:
            return False

        # buang keyword non-roadmap
        EXCLUDE_PATH = [
            "best-practices",
            "questions",
            "guide",
            "performance",
            "test",
            "top"
        ]

        if any(word in path for word in EXCLUDE_PATH):
            return False

        return True
    except:
        return False

# FILTER TITLE
def clean_title(title):
    EXCLUDE = ["Performance", "Best", "Guide", "Questions"]
    return not any(word.lower() in title.lower() for word in EXCLUDE)

# FILTER SKILL
EXCLUDE_KEYWORDS = [
    "guide", "best", "question", "interview",
    "how to", "top", "quiz", "example",
    "lifecycle", "newsletter", "review",
    "complete", "introduction", "roadmap.sh"
]

def is_valid_skill(text):
    text = text.strip()
    text_lower = text.lower()

    if text == "":
        return False

    if any(word in text_lower for word in EXCLUDE_KEYWORDS):
        return False

    if len(text.split()) > 3:
        return False

    if any(char.isdigit() for char in text):
        return False

    return True

def normalize(text):
    return text.strip().title()

# STEP 1: AMBIL ROADMAP LIST
driver.get("https://roadmap.sh/")

wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "a")))
elements = driver.find_elements(By.TAG_NAME, "a")

roadmaps = []

for el in elements:
    try:
        title = el.text.strip()
        link = el.get_attribute("href")

        if (
            title != "" and
            link and
            is_common_role(title) and
            is_valid_roadmap_url(link) and
            clean_title(title)
        ):
            roadmaps.append({
                "title": title,
                "link": link
            })
    except:
        continue

# remove duplicate
unique_roadmaps = list({r['link']: r for r in roadmaps}.values())

print(f"Total roadmap (clean): {len(unique_roadmaps)}")

# STEP 2: SCRAPE TIAP ROADMAP
all_data = []

for roadmap in unique_roadmaps:
    title = roadmap["title"]
    link = roadmap["link"]

    print(f"Scraping: {title}")

    try:
        driver.get(link)

        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//*[name()='text']"))
        )

        nodes = driver.find_elements(By.XPATH, "//*[name()='text']")

        skills = []

        for n in nodes:
            txt = n.text.strip()

            if is_valid_skill(txt):
                skills.append(normalize(txt))

        skills = list(set(skills))

        all_data.append({
            "role": title,
            "skills": skills,
            "url": link
        })

    except Exception as e:
        print(f"Error di {title}: {e}")
        continue

# =========================
# STEP 3: SIMPAN DATA
# =========================
with open("roadmap_data_final.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, indent=2, ensure_ascii=False)

driver.quit()

print("DONE! Data bersih disimpan di roadmap_data_final.json")

Total roadmap (clean): 14
Scraping: Frontend
Scraping: Backend
Scraping: Full Stack
Scraping: DevOps
Scraping: Data Analyst
Scraping: AI Engineer
Scraping: AI and Data Scientist
Scraping: Data Engineer
Scraping: Machine Learning
Scraping: QA
Scraping: Cyber Security
Scraping: UX Design
Scraping: 26 Mar, 2026
Scala Roadmap, AI Engineer Review
Error di 26 Mar, 2026
Scala Roadmap, AI Engineer Review: Message: 

Scraping: 22 Aug, 2025
Data Engineering, Machine Learning Roadmap and more.
Error di 22 Aug, 2025
Data Engineering, Machine Learning Roadmap and more.: Message: 

✅ DONE! Data bersih disimpan di roadmap_data_final.json


In [ ]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# LOAD DATA
with open("json/roadmap_data_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# LOAD MODEL
model = SentenceTransformer('all-MiniLM-L6-v2')

# CATEGORY (EXPANDED)
GENERAL_CATEGORIES = {
    "fundamental": "html css javascript dom browser basics",
    "programming": "javascript typescript nodejs python java programming",
    "tools": "git github gitlab npm yarn cli tools",
    "framework": "react vue angular nextjs framework frontend backend",
    "database": "sql mysql postgresql mongodb redis database",
    "architecture": "system design microservices scalability distributed systems",
    "testing": "unit testing integration testing automation testing",
    "security": "authentication authorization jwt oauth owasp security",
    "devops": "deployment docker kubernetes ci cd cloud aws gcp",
    "advanced": "performance optimization api caching profiling"
}

# RULE OVERRIDE (STRONG)
RULE_OVERRIDE = {
    # fundamental
    "Html": "fundamental",
    "Css": "fundamental",
    "Javascript": "fundamental",

    # programming
    "Typescript": "programming",
    "Nodejs": "programming",
    "Python": "programming",

    # tools
    "Git": "tools",
    "Github": "tools",
    "Gitlab": "tools",
    "Npm": "tools",
    "Yarn": "tools",

    # framework
    "React": "framework",
    "Vue.Js": "framework",
    "Angular": "framework",
    "Next.Js": "framework",
    "React-Router": "framework",
    "Solid Js": "framework",

    # database
    "Mysql": "database",
    "Postgresql": "database",
    "Mongodb": "database",
    "Redis": "database",

    # devops
    "Docker": "devops",
    "Kubernetes": "devops",
    "Aws": "devops",
    "Deployment": "devops",

    # security
    "Jwt": "security",
    "Oauth": "security",
    "Web Security": "security",

    # testing
    "Testing": "testing",
    "Vitest": "testing",
    "Unit Testing": "testing",
    "Integration Testing": "testing",

    # advanced
    "Web Apis": "advanced"
}

# CLEANING
EXCLUDE = [
    "roadmap", "project", "ideas",
    "guide", "learn", "what is",
    "how", "recommendation"
]

NOISE = [
    "ai assisted",
    "implementing",
    "in development",
    "prompt",
    "agents"
]

def clean_noise(skills):
    return [
        s for s in skills
        if not any(word in s.lower() for word in EXCLUDE + NOISE)
    ]

# AI + RULE CATEGORIZATION
def categorize_ai(skills):
    category_names = list(GENERAL_CATEGORIES.keys())
    category_desc = list(GENERAL_CATEGORIES.values())

    category_vectors = model.encode(category_desc)
    skill_vectors = model.encode(skills)

    result = {k: [] for k in category_names}
    result["others"] = []

    for i, skill_vec in enumerate(skill_vectors):
        skill = skills[i]

        # RULE PRIORITY
        if skill in RULE_OVERRIDE:
            result[RULE_OVERRIDE[skill]].append(skill)
            continue

        similarities = cosine_similarity([skill_vec], category_vectors)[0]
        best_idx = similarities.argmax()
        best_score = similarities[best_idx]

        if best_score < 0.35:
            result["others"].append(skill)
        else:
            result[category_names[best_idx]].append(skill)

    return result

# SMART ORDER
ORDER = [
    "fundamental",
    "programming",
    "tools",
    "framework",
    "database",
    "architecture",
    "testing",
    "security",
    "devops",
    "advanced"
]

# PRIORITY (biar urutan natural)
PRIORITY_ORDER = {
    "Html": 1,
    "Css": 2,
    "Javascript": 3,
    "Git": 4,
    "Npm": 5,
    "Typescript": 6,
    "React": 7,
    "Next.Js": 8,
    "Web Apis": 9,
    "Testing": 10,
    "Deployment": 11
}

def smart_sort(skills):
    return sorted(skills, key=lambda x: PRIORITY_ORDER.get(x, 999))

def flatten(categorized):
    result = []

    for cat in ORDER:
        skills = categorized.get(cat, [])

        if cat == "fundamental":
            skills = smart_sort(skills)

        result.extend(skills)

    return result

# MAIN PROCESS
final_data = []

for item in data:
    role = item["role"]
    skills = item["skills"]

    # 1. Clean
    skills = clean_noise(skills)

    # 2. Categorize
    categorized = categorize_ai(skills)

    # 3. Build learning path
    learning_path = flatten(categorized)

    final_data.append({
        "role": role,
        "learning_path": learning_path,
        "categorized": categorized
    })

# SAVE
with open("json/roadmap_ai_super_final.json", "w", encoding="utf-8") as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print("DONE! SUPER FINAL ROADMAP GENERATED")

DONE! SUPER FINAL ROADMAP GENERATED


In [11]:
import csv
import json

with open("json/roadmap_ai_fixed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("csv/roadmap_final.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # header lengkap
    writer.writerow(["role", "step", "skill", "category"])

    for item in data:
        role = item["role"]
        learning_path = item["learning_path"]
        categorized = item["categorized"]

        # buat mapping skill → category
        skill_to_category = {}
        for cat, skills in categorized.items():
            for s in skills:
                skill_to_category[s] = cat

        # tulis per skill
        for i, skill in enumerate(learning_path, start=1):
            category = skill_to_category.get(skill, "unknown")
            writer.writerow([role, i, skill, category])

print("CSV lengkap berhasil dibuat")

CSV lengkap berhasil dibuat


In [12]:
import pandas as pd
roadmap_df = pd.read_csv("csv/roadmap_final.csv")

In [17]:
roadmap_df[roadmap_df["role"] == "Frontend"]

,role,step,skill,category
0,Frontend,1,Html,fundamental
1,Frontend,2,Css,fundamental
2,Frontend,3,Javascript,fundamental
3,Frontend,4,Css Frameworks,fundamental
4,Frontend,5,Html Templates,fundamental
5,Frontend,6,Shadow Dom,fundamental
6,Frontend,7,Nodejs,programming
7,Frontend,8,Typescript,programming
8,Frontend,9,Yarn,tools
9,Frontend,10,Github,tools
